# Phân công công việc tuần 12

## 1. Idk-HuyNguyxn

### a. Sửa lỗi biên dịch
- Lớp AudioPlayer thiếu phương thức setVolume của interface Player $ \rightarrow $ Thêm phương thức này (logic có thể thêm sau)

### b. Tích hợp javafx.scene.media.MediaPlayer
Dùng MediaPlayer của JavaFX để phát nhạc thật thay vì thay đổi các cờ Boolean.
- Import các thư viện cần thiết
- Thêm thuộc tính: private MediaPlayer mediaPlayer;

In [ ]:
import javafx.scene.media.Media;
import javafx.scene.media.MediaPlayer;
import javafx.util.Duration;
import java.io.File;

public class AudioPlayer implements Player {
    // ... (các thuộc tính cũ) ...
    private MediaPlayer mediaPlayer; // Đối tượng phát nhạc thực sự
    // ...
}

### c. Cài đặt lại các phương thức điều khiển
Thay thế logic mô phỏng trong các hàm @Override bằng cách gọi đến mediaPlayer: play(), pause(), stop(), seek(int seconds), setVolume(double volume)

### d. Xử lý Callback (khi hết bài)
- Trong hàm play() (hoặc khi tạo MediaPlayer), gắn callback onSongEnd (mà AudioPlayer đang lưu trữ) vào MediaPlayer thật.
- (Logic của onSongEnd thường sẽ là gọi next(), việc này sẽ do MainController hoặc lớp điều phối quyết định và setOnSongEnd cho AudioPlayer).

In [ ]:
// Khi tạo MediaPlayer
mediaPlayer.setOnEndOfMedia(() -> {
    if (onSongEnd != null) {
        onSongEnd.run(); // Gọi callback đã được set từ bên ngoài
    }
});

### e. Cung cấp dữ liệu cho Giao diện (UI Binding)
AudioPlayer phải cung cấp các thuộc tính (Properties) mà MediaPlayer có:
- mediaPlayer.currentTimeProperty(): Trả về một ReadOnlyDoubleProperty cho UI biết thời gian hiện tại.
- mediaPlayer.totalDurationProperty(): Trả về một ReadOnlyDoubleProperty cho UI biết tổng thời gian bài hát.
- mediaPlayer.statusProperty(): Trả về trạng thái (Playing, Paused, Stopped...) để UI cập nhật icon Play/Pause.

# 2. Try-Harder-2day

## a. Sửa đổi PlaybackControl để "Điều khiển" (Control) Player
- Refactor PlaybackControl: Thêm targetPlayer (Player) làm thuộc tính.
- Cần thêm một tham chiếu đến Player (interface) và gọi các phương thức của Player đó.

In [ ]:
package com.musicPlayer;

public class PlaybackControl {
    
    // --- THÊM MỚI ---
    private Player targetPlayer; // Tham chiếu đến trình phát nhạc thực tế

    // --- SỬA ĐỔI ---
    // Constructor (hoặc thêm 1 hàm set) để nhận Player
    public PlaybackControl(Player targetPlayer) {
        this.targetPlayer = targetPlayer; // Nhận tham chiếu Player
        resetAll();
    }
    
    // Hàm set dự phòng nếu không dùng constructor
    public void setTargetPlayer(Player targetPlayer) {
        this.targetPlayer = targetPlayer;
    }
    // ---------------

    // ... (các hàm get giữ nguyên) ...

    // --- SỬA ĐỔI CÁC HÀM SET/TOGGLE ---
    
    public int setVolume(int volume) {
        if (volume < 0) this.volume = 0;
        else if (volume > 100) this.volume = 100;
        else this.volume = volume;
        
        // **GỌI PLAYER:** Chuyển đổi (0-100) -> (0.0-1.0)
        if (targetPlayer != null) {
            targetPlayer.setVolume(this.volume / 100.0); 
        }
        return this.volume;
    }
    
    public int changeVolume(int amount) {
        setVolume(this.volume + amount); // Tận dụng logic của setVolume
        return this.volume;
    }
    
    public boolean toggleMute() {
        isMuted = !isMuted;
        
        // **GỌI PLAYER:** (Giả sử Player có hàm setMute)
        // if (targetPlayer != null) {
        //     targetPlayer.setMute(isMuted); 
        // }
        
        // Nếu Player không có setMute, có thể tạm thời set volume về 0
        if (targetPlayer != null) {
            if (isMuted) targetPlayer.setVolume(0.0);
            else targetPlayer.setVolume(this.volume / 100.0);
        }
        
        return isMuted;
    }

    public double setSpeed(double speed) {
        // ... (logic giới hạn 0.5-2.0 giữ nguyên) ...
        this.playbackSpeed = speed; // Cần đảm bảo logic giới hạn đã chạy
        
        // **GỌI PLAYER:** (Giả sử Player có hàm setSpeed)
        // if (targetPlayer != null) {
        //     targetPlayer.setSpeed(this.playbackSpeed); 
        // }
        
        return this.playbackSpeed;
    }

    public boolean toggleShuffle() {
        isShuffle = !isShuffle;
        
        // **GỌI PLAYER:** (Giả sử Player có hàm setShuffle)
        // if (targetPlayer != null) {
        //     targetPlayer.setShuffle(isShuffle); 
        // }
        return isShuffle;
    }

    public boolean toggleRepeat() {
        isRepeat = !isRepeat;
        
        // **GỌI PLAYER:** (Giả sử Player có hàm setRepeat)
        // if (targetPlayer != null) {
        //     targetPlayer.setRepeat(isRepeat); 
        // }
        return isRepeat;
    }

    // ... (các hàm khác giữ nguyên) ...
}

## b. Kết hợp với Idk-HuyNguyxn
- Player.java (interface) và AudioPlayer.java hiện tại chưa có các phương thức như setSpeed, setMute, setShuffle, setRepeat $\rightarrow$ Trao đổi để thêm các phương thức còn thiếu này vào interface Player và lớp AudioPlayer.
- Idk-HuyNguyxn phải cài đặt các hàm mới này trong AudioPlayer (dù chỉ là lưu giá trị hoặc gọi mediaPlayer.setRate(speed)).

In [ ]:
public interface Player {
    // ... (các hàm cũ) ...
    void setVolume(double volume); // Đã có

    // --- CẦN THÊM MỚI ---
    void setMute(boolean mute); 
    void setSpeed(double speed);
    void setShuffle(boolean shuffle);
    void setRepeat(boolean repeat);
    
    // ... (các hàm isPlaying... giữ nguyên) ...
}

## c. Tích hợp Timer
- Đảm bảo Idk-HuyNguyxn implements TimerListener.
- Hướng dẫn nhóm UI cách kết nối Timer và AudioPlayer.

# 3. cacancap hoặc Neucromancer

## a. Hoàn thiện Song.java
- Override equals() và hashCode(): Vì Song được dùng làm key trong HashMap của lớp History. Nếu không có, History sẽ bị sai.

In [ ]:
@Override
public boolean equals(Object o) {
    if (this == o) return true; // Nếu là cùng 1 đối tượng
    if (o == null || getClass() != o.getClass()) return false; // Nếu khác kiểu
    Song song = (Song) o;
    // Coi 2 bài hát là một nếu ID giống nhau (đây là cách tốt nhất)
    return java.util.Objects.equals(songID, song.songID); 
}

@Override
public int hashCode() {
    // Chỉ hash dựa trên ID
    return java.util.Objects.hash(songID);
}

## b. Đồng bộ đơn vị thời gian
- Song.duration đang là int (giây), nhưng Song.timeStamps lại là List<Integer> (milliseconds). Lớp AudioPlayer cũng đang dùng double currentTime.
- Cần thống nhất với Idk-HuyNguyxn về đơn vị thời gian
- Đề xuất: Thống nhất dùng giây (seconds) và dùng kiểu double để có độ chính xác cao (cho cả duration và currentTime trong AudioPlayer).

In [ ]:
// private int duration; // Thay đổi
private double duration; // Đổi thành double (giây, có thể lẻ)

// Sửa các constructor
public Song(String title, String artist, String album, double duration, String url) {
    // ...
    this.duration = duration;
    // ...
}

// Sửa getter/setter
public double getDuration() { return duration; }
public void setDuration(double duration) { this.duration = duration; }

// Sửa getIndexAtTime để nhận double (giây)
public int getIndexAtTime(double currentTimeInSeconds) {
    if (timeStamps == null || timeStamps.isEmpty()) return -1;

    // Chuyển currentTime (giây) sang milliseconds để so sánh
    int currentTimeInMillis = (int) (currentTimeInSeconds * 1000); 

    if (currentTimeInMillis < 0 || currentTimeInMillis > (this.duration * 1000)) return -1;

    // Sửa logic so sánh, đảm bảo timeStamps (milliseconds)
    if (currentTimeInMillis < timeStamps.get(0)) return 0; // Chưa tới lời đầu tiên

    for (int i = 0; i < timeStamps.size(); i++) {
        int start = timeStamps.get(i); // start (ms)
        // Chuyển duration (giây) sang ms
        int end = (i + 1 < timeStamps.size()) ? timeStamps.get(i + 1) : (int)(this.duration * 1000); 

        if (currentTimeInMillis >= start && currentTimeInMillis < end)
            return i;
    }
    return timeStamps.size() - 1; // Mặc định là lời cuối cùng nếu vượt quá
}

## c. Hoàn thiện Playlist.java:
- Sửa getSongs() để trả về bản sao, ngăn chặn lớp bên ngoài sửa đổi trực tiếp list.

In [ ]:
public List<Song> getSongs(){
    // Trả về một bản sao (copy) của danh sách
    return new ArrayList<>(this.songs);
    // Hoặc trả về bản chỉ đọc:
    // return Collections.unmodifiableList(this.songs);
}

## d. Xóa println (Dọn dẹp):
- Rà soát lại Song.java (và Playlist.java), xóa các phương thức chỉ dùng để in ra console như displayLyricLine, displayAllLyrics, displayRating.
- Thay thế (Nếu cần): Có thể thay displayRating bằng getAverageRating() (trả về double). Thay displayAllLyrics bằng getLyricsWithTimestamps() (trả về String hoặc List<String>).

# 4.	PhamHoa2006

## a. Hoàn thiện User.java (Bổ sung tính năng):
- Thêm lại thuộc tính private History history = new History(); và public History getHistory().
- Thêm lại thuộc tính private List<User.SharedItem> sharedInbox = new ArrayList<>(); (và inner class SharedItem, cùng các phương thức receiveShared..., viewInbox).
- Sửa lỗi chính tả (descripsion -> description) và kiểu dữ liệu (age -> int).
- Thêm phương thức checkPassword(String password) để UserManager sử dụng (mặc dù UserManager đã tự hash được, nhưng nên để User tự kiểm tra pass của mình).

In [ ]:
// Trong User.java
public boolean checkPassword(String password) {
    if (password == null) return false;
    return this.passwordHash.equals(hashPassword(password));
}

## b. Hoàn thiện UserManager.java (Thêm trạng thái và lưu trữ):
- Thêm currentUser:

In [ ]:
// Trong UserManager.java
private User currentUser = null; // Thêm dòng này (không cần static nếu UserManager là Singleton)

// Sửa login()
public boolean login(String username, String password) {
    // ... (code kiểm tra user/pass như cũ) ...
    if (tmp == null || !tmp.checkPassword(password)) { // Dùng tmp.checkPassword
        System.out.println("Ten dang nhap hoac mat khau khong hop le");
        currentUser = null; // Đảm bảo currentUser là null nếu sai
        return false;
    }
    currentUser = tmp; // *** LƯU USER ĐANG ĐĂNG NHẬP ***
    System.out.println("Dang nhap thanh cong: " + currentUser.getUsername());
    return true;
}

// Thêm logout()
public void logout() {
    System.out.println("Tam biet: " + (currentUser != null ? currentUser.getUsername() : ""));
    currentUser = null;
}

// Thêm getter
public User getCurrentUser() {
    return currentUser;
}

- Áp dụng Singleton Pattern (Bắt buộc): Chuyển HashMap thành non-static và thêm Singleton (như code repo cũ) để đảm bảo chỉ có 1 UserManager.

In [ ]:
// Trong UserManager.java
private HashMap<String, User> users = new HashMap<>(); // Bỏ static
private User currentUser = null; // Biến instance

private static UserManager instance = null; // Biến static cho Singleton

private UserManager() {} // Constructor private

public static UserManager getInstance() {
    if (instance == null) {
        instance = new UserManager();
        // instance.loadDataFromDisk(); // Sẽ thêm ở bước 3
    }
    return instance;
}

// Sửa tất cả các phương thức khác (findUser, register...) thành non-static (bỏ static)
public User findUser(String username) { ... }
public boolean register(String username, String password) { ... }
// ... (và tất cả các hàm khác)

- Viết hàm Lưu/Tải dữ liệu (Persistence): Viết 2 phương thức saveDataToDisk() và loadDataFromDisk() sử dụng Serialization (vì HashMap đã implements Serializable).

In [ ]:
// Trong UserManager.java (đã là Singleton)
// Thêm import java.io.*;

private static final String DATA_FILE = "userdata.dat"; // Tên file lưu

// Gọi hàm này khi khởi tạo trong getInstance()
public void loadDataFromDisk() {
    try (ObjectInputStream ois = new ObjectInputStream(new FileInputStream(DATA_FILE))) {
        // Đọc HashMap từ file
        this.users = (HashMap<String, User>) ois.readObject(); 
        System.out.println("Tai du lieu user thanh cong.");
    } catch (FileNotFoundException e) {
        System.out.println("Khong tim thay file du lieu cu, khoi tao moi.");
        this.users = new HashMap<>();
    } catch (IOException | ClassNotFoundException e) {
        System.out.println("Loi khi doc file, khoi tao moi.");
        this.users = new HashMap<>();
        e.printStackTrace();
    }
}

// Gọi hàm này khi đóng ứng dụng (trong Main.java)
public void saveDataToDisk() {
    try (ObjectOutputStream oos = new ObjectOutputStream(new FileOutputStream(DATA_FILE))) {
        // Ghi HashMap hiện tại vào file
        oos.writeObject(this.users); 
        System.out.println("Luu du lieu user thanh cong.");
    } catch (IOException e) {
        System.out.println("Loi khi luu file.");
        e.printStackTrace();
    }
}

// Đảm bảo User.java và các lớp nó chứa (Playlist, Song, History...) đều implements Serializable

## c. Phối hợp
- Đảm bảo User.java (và các lớp nó chứa như Song, Playlist, History) đều implements Serializable để logic lưu/tải hoạt động.

# 5. pvq-vn

# 6. cacancap hoặc Neucromancer

## a. Viết code @FXML để "bắt" các fx:id mà pvq-vn đã tạo.
## b. Tích hợp
- Gọi audioPlayer.play() của Idk-HuyNguyxn
- Gọi playbackControl.setVolume() của Try-Harder-2day
- Gọi userManager.login() của PhamHoa2006
- Hiển thị FileChooser (nút Tải lên) và gọi playlist.addSong().
- Gọi history.addSong(...) khi phát nhạc.
- Gọi recommendationEngine.suggest...() khi nhấn nút Gợi ý.
- Gọi timer.setTimer(...) khi người dùng hẹn giờ.
- Gọi shareService.shareSongToUser(...) khi nhấn nút Chia sẻ.
## c. Cập nhật UI
- Lấy dữ liệu trả về từ các hàm backend (ví dụ history.getUserHistoryByDate()) và cập nhật lên các TableView/ListView tương ứng trên giao diện.